# Comparative Study of Deep Learning Architectures for Khmer ASR
### Complete Google Colab Experiment & Retraining Pipeline

**Course**: Deep Learning Final Project (2026–2027)  
**Lecturer**: Mr. Soklong HIM  
**Hardware**: Google Colab Tesla T4 GPU  
**Dataset**: Google FLEURS Khmer (`km_kh`)  

### 3 Approaches Compared:
1. **Approach 1**: OpenAI Whisper-Tiny Full Fine-Tuning (Seq2Seq Transformer)
2. **Approach 2**: Meta MMS-1B Khmer CTC (Non-Autoregressive Acoustic CTC Model)
3. **Approach 3 (Ablation)**: Whisper-Tiny Frozen Encoder (Linear Probe / Transfer Learning)

## 1. Hardware & GPU Check
Make sure your Colab session has a GPU assigned: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: Running on CPU. Please switch to T4 GPU in Runtime settings.')

## 2. Install Required Dependencies & Free Disk Space

In [ ]:
# Clean any old cached downloads to avoid filling Colab disk
!rm -rf /content/hf_cache /root/.cache/huggingface
!pip install -q torch torchaudio transformers datasets evaluate jiwer accelerate tensorboard soundfile librosa matplotlib python-pptx

## 3. Clone Repository or Upload Code
If your repository is on GitHub, uncomment Option A. Otherwise, run Option B to upload your `src/`, `slides/`, and `samples/` folders.

In [ ]:
# Option A: Clone your GitHub repository (Recommended)
!git clone https://github.com/Seypa-47/khmer_asr.git
%cd khmer_asr

# Option B: If not using git, upload src/train_mms.py manually:
# from google.colab import files
# uploaded = files.upload()
# !mkdir -p src
# !mv -f *.py src/ 2>/dev/null || true


## 4. Run Approach 1: High-Accuracy Whisper Fine-Tuning (10 Epochs)
Trains `openai/whisper-tiny` on Google FLEURS `km_kh` (~1.5 GB). Uses `--skip-ddd` so it does NOT download the 60 GB DDD dataset, preventing disk quota errors.

In [ ]:
!python src/finetune_whisper.py \
  --output-dir "./models/whisper-tiny-khmer" \
  --model-name "openai/whisper-tiny" \
  --use-fleurs-train \
  --skip-ddd \
  --seed 42 \
  --num-train-epochs 10 \
  --learning-rate 1e-4 \
  --warmup-steps 200 \
  --per-device-train-batch-size 8 \
  --per-device-eval-batch-size 8 \
  --gradient-accumulation-steps 2 \
  --logging-steps 25 \
  --eval-steps 100 \
  --save-steps 100 \
  --fp16

## 5. Run Approach 2: Meta MMS-1B Khmer (CTC Acoustic Model)
Fine-tunes the 1-Billion parameter Wav2Vec 2.0 acoustic backbone with Khmer (`khm`) CTC vocabulary adapter.

In [ ]:
!python src/train_mms.py \
  --output-dir "./models/mms-khmer-ctc" \
  --model-id "facebook/mms-1b-all" \
  --target-lang "khm" \
  --seed 42 \
  --max-train-samples 1000 \
  --max-eval-samples 200 \
  --max-test-samples 200 \
  --num-train-epochs 15 \
  --learning-rate 5e-5 \
  --unfreeze-top-layers 4 \
  --apply-spec-augment \
  --lr-scheduler-type cosine \
  --warmup-ratio 0.1 \
  --per-device-train-batch-size 1 \
  --per-device-eval-batch-size 1 \
  --gradient-accumulation-steps 8 \
  --eval-steps 50 \
  --save-steps 50 \
  --fp16


## 6. Run Approach 3 (Ablation): Whisper Frozen Encoder (Linear Probe)
Freezes the 9.3M parameter Whisper audio encoder and tunes only the decoder to evaluate acoustic transferability.

In [ ]:
!python src/finetune_whisper.py \
  --output-dir "./models/whisper-tiny-khmer-frozen" \
  --model-name "openai/whisper-tiny" \
  --use-fleurs-train \
  --skip-ddd \
  --freeze-encoder \
  --seed 42 \
  --num-train-epochs 5 \
  --learning-rate 1e-4 \
  --warmup-steps 100 \
  --per-device-train-batch-size 8 \
  --per-device-eval-batch-size 8 \
  --gradient-accumulation-steps 2 \
  --logging-steps 25 \
  --eval-steps 100 \
  --save-steps 100 \
  --fp16

## 7. Generate Rubric Deliverables, Figures & Slide Deck
Generates `results/learning_curves.png`, `results/metrics_comparison.png`, `results/summary_table.md`, and `slides/khmer_asr_presentation.pptx`.

In [ ]:
# Generate evaluation curves and tables
!python src/evaluate_and_plot.py

# Generate 13-slide PowerPoint presentation
!python slides/generate_slides.py

# Display figures directly in Colab
from IPython.display import Image, display
print('\n--- Learning Curves ---')
display(Image('results/learning_curves.png'))
print('\n--- Approaches CER Comparison ---')
display(Image('results/metrics_comparison.png'))

## 8. Package and Download Trained Model, Results & Presentation
Cleans intermediate optimizer weights and downloads a single zip containing:
- `models/whisper-tiny-khmer/` (clean model weights ready for `app.py`)
- `results/` (figures and summary tables)
- `slides/` (PowerPoint presentation)

In [ ]:
# Clean heavy optimizer checkpoints
!rm -rf models/*/checkpoint-* models/*/runs 2>/dev/null || true
# Package all trained models (Whisper + MMS), results, and slides
!zip -r khmer_asr_trained_bundle.zip models/ results/ slides/
from google.colab import files
files.download('khmer_asr_trained_bundle.zip')
